# Shen et al. (2023) TATN — Tables II, IV, V, VI, VII

Paper-faithful Panasonic NCR18650PF temperature-transfer reproduction. The output compares only **Source pretrained** and **TATN**. Target test labels are evaluated once after training and never select a checkpoint.

The default `screening` mode cheaply produces all five table shapes and every temperature-pair comparison from two representative folds. Use `optimized_full` for all nine folds with early stopping, or `paper_full` for the released repository's fixed 2000-epoch default. Every mode has a separate Drive output and cache.

Before running, add a Colab Secret named `GITHUB_TOKEN`, grant it read access to the private repository, and enable Notebook access for the secret.

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from urllib.parse import urlunsplit

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
except ImportError:
    raise RuntimeError('This setup cell is intended for Google Colab') from None

REPO = Path('/content/battery-soc-transfer-benchmark')
REPO_URL = urlunsplit(('https', 'github.com', '/zhangcastle/battery-soc-transfer-benchmark.git', '', ''))
github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Add the GITHUB_TOKEN Colab Secret and enable Notebook access')
if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'Remove incomplete checkout and rerun: {REPO}')

with tempfile.TemporaryDirectory() as temp_dir:
    askpass = Path(temp_dir) / 'git-askpass.sh'
    askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) echo "x-access-token" ;;\n*) printf "%s" "$GITHUB_TOKEN" ;;\nesac\n')
    askpass.chmod(0o700)
    git_env = os.environ.copy()
    git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
    command = (['git', '-C', str(REPO), 'pull', '--ff-only'] if (REPO / '.git').exists() else ['git', 'clone', '--depth', '1', REPO_URL, str(REPO)])
    result = subprocess.run(command, env=git_env, text=True, capture_output=True)
    git_env.pop('GITHUB_TOKEN', None)
del github_token
if result.returncode:
    raise RuntimeError('Authenticated GitHub setup failed; check the token repository permission and Notebook access') from None
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
os.chdir(REPO)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
import battery_benchmark
print('Repository ready:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('Package:', battery_benchmark.__file__)

## Protocol

- Five temperatures: 25, 10, 0, -10, -20 °C; nine drive cycles each.
- Every tenth raw sample; non-overlapping windows of 90.
- Per-temperature min-max fitted to all nine cycles, exactly as stated in the paper.
- Table II: nine-fold leave-one-cycle-out per temperature; report the eight-cycle training error averaged across folds.
- Transfer source models: train on all nine source-temperature cycles.
- CNN(256×2) + four released-repo BiLSTM layers, hidden size 64, dropout 0.2.
- Adam, lr 1e-3, batch 64. The paper does not state epochs; 2000 is only the released-repo CLI default.
- Screening: Cycle_4 + UDDS folds, at most 150 epochs, minimum 30, patience 15. Diagnostic only.
- Optimized full: all nine folds, at most 300 epochs, minimum 60, patience 30.
- Semi-supervised: one test cycle, one non-test cycle with its first 50% labels, seven pseudo-labelled cycles.
- Unsupervised: one test cycle and eight pseudo-labelled cycles.

In [ ]:
import torch
from battery_benchmark.utils.config import load_yaml
from battery_benchmark.reproductions.shen_2023.data import PROFILES, TEMPERATURES, load_temperature_cycles
from battery_benchmark.reproductions.shen_2023.suite import PAPER_TABLES, ShenTableSuite

PATHS = load_yaml(REPO / 'configs/paths/colab_drive.yaml')
RAW_ROOT = Path(PATHS['raw_data_root']) / 'Panasonic_18650PF'
RUN_MODE = 'screening'  # screening | optimized_full | paper_full
PAPER_FULL = RUN_MODE == 'paper_full'
SCREENING = RUN_MODE == 'screening'
OUTPUT_NAME = {'screening': 'tatn_screening', 'optimized_full': 'tatn_optimized_protocol', 'paper_full': 'tatn_paper_protocol'}[RUN_MODE]
OUTPUT_DIR = Path(PATHS['output_root']) / 'reproductions/shen_2023' / OUTPUT_NAME
EPOCHS = 2000 if PAPER_FULL else (150 if SCREENING else 300)
TRANSFER_EPOCHS = EPOCHS
RUN_PROFILES = ('Cycle_4', 'UDDS') if SCREENING else PROFILES
EARLY_STOPPING_PATIENCE = None if PAPER_FULL else (15 if SCREENING else 30)
MIN_EPOCHS = 0 if PAPER_FULL else (30 if SCREENING else 60)
assert RAW_ROOT.exists(), f'Missing Panasonic data: {RAW_ROOT}'
print('mode:', RUN_MODE)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('output:', OUTPUT_DIR)
print('profiles:', RUN_PROFILES)

In [ ]:
for temperature in TEMPERATURES:
    cycles = load_temperature_cycles(RAW_ROOT, temperature)
    print(f'{temperature:>3}C:', {name: len(cycle.soc) for name, cycle in cycles.items()})

In [ ]:
suite = ShenTableSuite(
    RAW_ROOT, OUTPUT_DIR, epochs=EPOCHS, transfer_epochs=TRANSFER_EPOCHS,
    batch_size=64, learning_rate=1e-3, lambdas=(1.0, 1.0, 1.0),
    seed=100, device='auto', early_stopping_patience=EARLY_STOPPING_PATIENCE,
    min_epochs=MIN_EPOCHS, min_delta=1e-4,
)
table_ii = suite.run_table_ii(profiles=RUN_PROFILES)
display(table_ii.round(3).style.set_caption('Table II — source pretraining errors (%)'))
print('Paper Table II:', PAPER_TABLES['II'])

In [ ]:
# Resumable: each completed fold is written to OUTPUT_DIR/cache/<protocol-id>/folds.
suite.run_all_transfers(profiles=RUN_PROFILES)
print('Transfer folds complete or restored from cache:', len(suite.records))

In [ ]:
tables = suite.tables(profiles=RUN_PROFILES)
titles = {
    'II': 'Table II — fixed-temperature source pretraining',
    'IV': 'Table IV — 10C source, semi-supervised',
    'V': 'Table V — all sources, semi-supervised',
    'VI': 'Table VI — 10C source, unsupervised',
    'VII': 'Table VII — all sources, unsupervised',
}
for name in ('II', 'IV', 'V', 'VI', 'VII'):
    display(tables[name].round(3).style.set_caption(titles[name] + ' (MAE/RMSE %)'))
    print('Paper TATN reference:', PAPER_TABLES[name])
print('CSV outputs:', sorted(str(path) for path in OUTPUT_DIR.glob('table_*.csv')))

## Reading the result

All modes output Tables II/IV/V/VI/VII and compare only `Source pretrained` with `TATN`. Screening keeps every temperature pair but averages two folds, so it is a directional check rather than a paper reproduction.